# Chart patterns as model input

Do the 212 Bulkowski detectors in `app/python/patterns.py` predict anything?

One row per (symbol, bar). Features are how long ago each pattern completed. The
target is the return of a long trade closed by a trailing average measured from
the entry bar. Only the pattern calculation is imported — no engine, no
`PatternsStrategy` trading logic.

## Setup

In [ ]:
import bisect
import multiprocessing as mp
import os
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr

# The notebook lives in tools/ but every path in this project is repo-root
# relative, so walk up and work from there.
ROOT = pathlib.Path.cwd()
while not (ROOT / "app" / "python" / "patterns.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "app" / "python"))

import patterns as pat

DATA = "app/data/us_ndx100_2020_1d.parquet"    # or "app/data/bist_1d.parquet"

LOOKBACK = 500      # bars of history handed to each detector
WARMUP = 260        # first bar evaluated; the long patterns cannot form below this
MIN_BARS = 400      # a symbol needs this much history to enter the universe
GAP = 2             # a shape absent this many bars or fewer is the same occurrence

SMA_WINDOW = 20     # trailing average that closes the trade
MIN_HOLD = 5        # no exit before this
MAX_HOLD = 120      # hard cap; in practice it almost never binds

N_FOLDS = 5
EMBARGO = 10        # sessions between the last training exit and the test block
SEED = 7

SPECS = pat.PATTERNS
PARAMS = pat.PatternsStrategy    # detectors read their ~177 thresholds off the class
JOBS = max(1, (os.cpu_count() or 4) - 2)
MARKET = pathlib.Path(DATA).stem
CACHE = pathlib.Path("app/reports/pattern-model")
CACHE.mkdir(parents=True, exist_ok=True)

print(f"{len(SPECS)} detectors | market={MARKET} | jobs={JOBS}")

## Features

`<pattern>__age` — bars since that pattern last completed, `NaN` before its first
occurrence for that symbol.

Two things make that well defined. A detector answers "is this shape present in
my window", not "did it just complete", so consecutive bars repeat the same
detection; those are collapsed into runs and the run's first bar is the
completion. And a shape absent for `GAP` bars or fewer resumes its run rather
than starting a new one.

No sign: every pattern fires in one direction, so the column identity already
carries it. No decay or window parameter either — a tree splits on thresholds, so
`exp(-age/tau)` and `age <= k` are monotone re-expressions of `age` and offer the
same splits, while raw bar counts let the tree pick a per-pattern threshold.

The sweep is the only slow step and it is cached to parquet.

In [ ]:
frame = pd.read_parquet(DATA).sort_values(["symbol", "timestamp"])
dates = np.sort(frame["timestamp"].unique())

books = {}
for sym, g in frame.groupby("symbol", sort=True):
    if len(g) < MIN_BARS:
        continue
    o, h, l, c, v = (g[k].to_numpy(dtype=float)
                     for k in ("open", "high", "low", "close", "volume"))
    books[sym] = dict(o=o, h=h, l=l, c=c, v=v, ts=g["timestamp"].to_numpy())
SYMBOLS = sorted(books)


def scan_symbol(sym):
    """Every (bar, pattern) the detectors fire on, for one symbol.

    Pivots are computed once over the whole series and sliced per window: a
    pivot at i is decided by bars i-2..i+2, so a window's pivots are the full
    series' pivots restricted and shifted. All 212 detectors then share one
    pivot computation. The forked workers inherit `books`, so the price arrays
    never cross a pickle boundary.
    """
    b = books[sym]
    o, h, l, c, v = b["o"], b["h"], b["l"], b["c"], b["v"]
    span = pat.PIVOT_SPAN
    peaks, valleys = pat._pivots(h, True), pat._pivots(l, False)
    rows = []
    for t in range(WARMUP, len(c)):
        a = max(0, t - LOOKBACK + 1)
        pk = [i - a for i in peaks[bisect.bisect_left(peaks, a + span):
                                   bisect.bisect_right(peaks, t - span)]]
        vl = [i - a for i in valleys[bisect.bisect_left(valleys, a + span):
                                     bisect.bisect_right(valleys, t - span)]]
        bars = pat.Bars(o[a:t + 1], h[a:t + 1], l[a:t + 1], c[a:t + 1], v[a:t + 1],
                        peaks=pk, valleys=vl)
        rows += [(t, pid) for pid, spec in enumerate(SPECS)
                 if spec.detect(bars, PARAMS) is not None]
    return pd.DataFrame(rows, columns=["bar", "pattern_id"]).assign(symbol=sym)


path = CACHE / f"detections-{MARKET}.parquet"
if path.exists():
    det = pd.read_parquet(path)
else:
    t0 = time.time()
    with mp.get_context("fork").Pool(JOBS) as pool:
        det = pd.concat(pool.map(scan_symbol, SYMBOLS), ignore_index=True)
    det = det.astype({"bar": "int32", "pattern_id": "int16"})
    det.to_parquet(path, index=False)
    print(f"swept in {time.time() - t0:.0f}s -> {path}")

# Collapse consecutive detections into runs. A run breaks only after more than
# GAP bars of absence, and that test reads the previous detection, never a later
# one.
d = det.sort_values(["symbol", "pattern_id", "bar"])
b = d["bar"].to_numpy()
k = d.groupby(["symbol", "pattern_id"], sort=False).ngroup().to_numpy()
runs = d[(k != np.r_[-1, k[:-1]]) | (b - np.r_[0, b[:-1]] > GAP + 1)]

# Forward-fill the most recent completion down each column. Run starts only
# increase, which is what makes `maximum.accumulate` the right fill.
AGE_COLS = [f"{s.name}__age" for s in SPECS]
by_symbol = {s: g for s, g in runs.groupby("symbol", sort=False)}
parts = []
for sym in SYMBOLS:
    n = len(books[sym]["c"]) - WARMUP
    first = np.full((n, len(SPECS)), -1, dtype=np.int32)
    r = by_symbol.get(sym)
    if r is not None:
        f = r["bar"].to_numpy() - WARMUP
        first[f, r["pattern_id"].to_numpy()] = f
    np.maximum.accumulate(first, axis=0, out=first)
    t = np.arange(n, dtype=np.int32)[:, None]
    parts.append(np.where(first >= 0, t - first, np.nan).astype(np.float32))

BM = pd.DataFrame(np.concatenate(parts), columns=AGE_COLS)
BM.insert(0, "symbol", [s for s in SYMBOLS for _ in range(len(books[s]["c"]) - WARMUP)])
BM.insert(1, "timestamp", np.concatenate([books[s]["ts"][WARMUP:] for s in SYMBOLS]))
del parts

print(f"{len(det):,} detections -> {len(runs):,} occurrences (GAP={GAP})")
print(f"{len(BM):,} rows x {len(AGE_COLS)} columns, "
      f"{100 * BM[AGE_COLS].isna().to_numpy().mean():.0f}% NaN, "
      f"median age {np.nanmedian(BM[AGE_COLS].to_numpy()):.0f} bars")

## Target

`y` is the market-relative return of one long trade per row:

- **Enter** at the open after the signal bar, and only if that bar closed at or
  above its `SMA_WINDOW` average — a bar already below it would exit on entry.
- **Exit** at the open after the first close below the trailing average, where
  the average is measured **from the entry bar** and no exit is allowed before
  `MIN_HOLD`. The rule is self-stopping, so there is no separate stop.
- **Subtract** the equal-weight market over that trade's own window, so a rising
  tape cannot flatter a long label.

Anchoring the average to the entry is what keeps the labels distinct. Under a
plain `SMA_20` the exit is a property of the symbol, so every entry inside one
leg shares it — 8.9 entries per exit, against 2.0 measured from entry.

`exit_ts` is when each row's outcome becomes known; holds vary per row, so the
folds purge on it.

In [ ]:
# Equal-weight market: mean cross-sectional daily return, compounded.
pc = frame.groupby("symbol", sort=False)["close"].pct_change()
MIDX = (1.0 + pc.groupby(frame["timestamp"]).mean().reindex(dates).fillna(0.0)).cumprod().to_numpy()

offset, n = {}, 0
for sym in SYMBOLS:
    offset[sym] = n
    n += len(books[sym]["c"]) - WARMUP

y = np.full(len(BM), np.nan)
hold = np.full(len(BM), np.nan)
exit_ts = np.full(len(BM), np.datetime64("NaT", "us"))

for sym in SYMBOLS:
    b = books[sym]
    c, o, ts, T = b["c"], b["o"], b["ts"], len(b["c"])
    sma = pd.Series(c).rolling(SMA_WINDOW).mean().to_numpy()
    cum = np.r_[0.0, np.cumsum(c)]
    dpos = np.searchsorted(dates, ts)
    for t in range(WARMUP, T - 1):
        if not c[t] >= sma[t]:               # NaN-safe: also skips the warmup
            continue
        e = t + 1
        limit = min(e + MAX_HOLD, T - 1)
        k = e
        while k < limit:
            if k - e + 1 >= MIN_HOLD:
                a = max(e, k - SMA_WINDOW + 1)
                if c[k] < (cum[k + 1] - cum[a]) / (k - a + 1):
                    break
            k += 1
        x = k + 1
        if x > T - 1:                        # outcome runs past the data: censored
            continue
        row = offset[sym] + t - WARMUP
        y[row] = (o[x] / o[e] - 1.0) - (MIDX[dpos[x]] / MIDX[dpos[e]] - 1.0)
        hold[row] = x - e
        exit_ts[row] = ts[x]

BM["y"], BM["hold"], BM["exit_ts"] = y, hold, exit_ts
BM = BM[np.isfinite(y)].reset_index(drop=True)

print(f"{len(BM):,} trades ({100 * len(BM) / len(y):.0f}% of bars)")
print(f"hold  : p50 {BM['hold'].median():.0f} bars, p90 {BM['hold'].quantile(0.9):.0f}")
print(f"y     : mean {100 * BM['y'].mean():+.2f}%  sd {100 * BM['y'].std():.2f}%  "
      f"win {100 * (BM['y'] > 0).mean():.0f}%")

## Train and evaluate

Expanding-window folds, purged and embargoed. A random split would be fatal:
with 88 symbols it puts the same date on both sides and one market-wide move
leaks across. A training row is admissible only if **its own trade closed**
before the test block opens, minus the embargo — so the purge reads `exit_ts`,
not a fixed horizon. The assertion is worth more than the fold counts.

- **rho** — out-of-sample Spearman between prediction and realised excess return.
- **top_decile** vs **base** — mean `y` of the best-ranked 10% of test trades
  against the mean over all of them. The economic read.

In [ ]:
XGB = dict(objective="reg:squarederror", tree_method="hist", max_depth=5,
           eta=0.05, subsample=0.8, colsample_bytree=0.6,
           min_child_weight=200, reg_lambda=5.0, nthread=JOBS, seed=SEED)
ROUNDS = 300

enter = np.searchsorted(dates, BM["timestamp"].to_numpy())
leave = np.searchsorted(dates, BM["exit_ts"].to_numpy())
edges = np.linspace(enter.min(), enter.max() + 1, N_FOLDS + 2).astype(int)

X, yv = BM[AGE_COLS], BM["y"].to_numpy()
rows = []
for i in range(1, N_FOLDS + 1):
    a, b = edges[i], edges[i + 1]
    train = leave < a - EMBARGO       # the trade must have closed before the block
    test = (enter >= a) & (enter < b)
    assert leave[train].max() + EMBARGO <= enter[test].min(), "purge violated"
    model = xgb.train(XGB, xgb.DMatrix(X[train], label=yv[train]), num_boost_round=ROUNDS)
    p = model.predict(xgb.DMatrix(X[test]))
    yt = yv[test]
    top = p >= np.quantile(p, 0.9)
    rows.append(dict(fold=f"{pd.Timestamp(dates[a]).date()}..{pd.Timestamp(dates[b - 1]).date()}",
                     n_train=int(train.sum()), n_test=int(test.sum()),
                     rho=spearmanr(p, yt).statistic,
                     top_decile=100 * yt[top].mean(), base=100 * yt.mean()))

RESULT = pd.DataFrame(rows).set_index("fold")
display(RESULT.round(4))
print(f"mean rho {RESULT['rho'].mean():+.4f}   sd {RESULT['rho'].std():.4f}   "
      f"{int((RESULT['rho'] > 0).sum())}/{len(RESULT)} folds positive")